# 냉장고를 부탁해 · 식재료 인식 모델 v4 (33종)

**시작 전에**: `런타임 → 런타임 유형 변경 → T4 GPU`, 왼쪽 열쇠 아이콘(보안 비밀)에 `ROBOFLOW_API_KEY` 등록.
드라이브 `내 드라이브/yorijori/qc/qc_cutouts.zip` 이 있어야 한다 (내 PC에서 `qc_local.py`로 만든 파일).

| 셀 | 내용 | 시간 |
|---|---|---|
| ① | 설치 · 드라이브 연결 | 1~2분 |
| ② | 재료 33종 정의 | 몇 초 |
| ③ | Roboflow 공개 데이터 받기 + 합치기 | 20~40분 |
| ④ | AI Hub 오려 낸 재료 풀기 | 2~5분 |
| ⑤ | 오려 낸 재료 확인 | 몇 초 |
| ⑥ | 배경에 합성 → YOLO 라벨 (6,000장) | 10~20분 |
| ⑦ | 학습 설정 파일 (data_v4.yaml) | 몇 초 |
| ⑧ | 학습 (v3에서 이어서) | 1~2시간 |
| ⑨ | 평가 | 몇 분 |
| ⑩ | 웹용 ONNX 내보내기 | 1분 |

세션이 끊기면 ①②③⑦을 다시 실행한 뒤 이어 간다 (④⑥ 결과는 드라이브에 남아 있어 다시 안 해도 된다).

## ① 설치 · 드라이브 연결

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime

from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, random, shutil, zipfile, time
import numpy as np, cv2
from pathlib import Path

WORK = '/content/drive/MyDrive/fridge_yolo'           # 학습 결과(가중치, ONNX) — v1~v3 와 같은 곳
ROOT = Path('/content/drive/MyDrive/yorijori/qc')     # AI Hub 오려 낸 재료·합성 사진
CUT  = ROOT / 'cutouts'
os.makedirs(WORK, exist_ok=True); ROOT.mkdir(parents=True, exist_ok=True); CUT.mkdir(exist_ok=True)
%cd /content

## ② 재료 33종

In [ ]:
# 인식할 재료 33종. 0~24는 v3와 같은 번호, 25부터 새 재료. 번호 순서를 바꾸면 기존 라벨이 틀어지므로 고정한다.
# 이 목록이 웹의 classes.json 이 된다. 고기·김치·밥처럼 포장 때문에 모양이 제각각인 재료는 넣지 않는다(직접 입력).
import json
CLASSES_V4 = [
    ('cabbage', '양배추'), ('green_onion', '대파'), ('onion', '양파'), ('potato', '감자'), ('carrot', '당근'),
    ('zucchini', '애호박'), ('cucumber', '오이'), ('radish', '무'), ('spinach', '시금치'), ('mushroom', '버섯'),
    ('tomato', '토마토'), ('chili_pepper', '고추'), ('garlic', '마늘'), ('egg', '계란'), ('tofu', '두부'),
    ('milk', '우유'), ('apple', '사과'), ('banana', '바나나'), ('lemon', '레몬'), ('bell_pepper', '파프리카'),
    ('eggplant', '가지'), ('broccoli', '브로콜리'), ('corn', '옥수수'), ('cheese', '치즈'), ('sweet_potato', '고구마'),
    ('napa_cabbage', '배추'), ('pear', '배'), ('mandarin', '귤'), ('orange', '오렌지'),
    ('lime', '라임'), ('strawberry', '딸기'), ('grape', '포도'), ('kiwi', '키위'),
]
classes = [{'id': i, 'en': en, 'ko': ko} for i, (en, ko) in enumerate(CLASSES_V4)]
NAMES = [c['en'] for c in classes]
json.dump(classes, open('/content/classes.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(len(classes), '종:', ', '.join(f"{c['id']}.{c['ko']}" for c in classes))

## ③ Roboflow 공개 데이터 받기 + 합치기
출력 마지막의 **재료별 박스 수**에서 25~32번(배추~키위)에 숫자가 붙었는지 본다. 배추는 0이 정상(④~⑥ AI Hub로 채움).

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
%cd /content
import json, random, re, shutil, glob, yaml
from pathlib import Path
from collections import Counter

ROBOFLOW_SOURCES = [
    # --- 냉장고 혼합 사진 (가장 중요) ---
    ('smart-fridge-mvi88', 'smart-fridge-co7ul', 6),
    ('recipe-recomendation-app', 'fridge-detection-merged', 4),
    ('bottles-0ds2b', 'fridge-qjtyj', 1),
    ('fridge-0keux', 'fridge-0fckr', 1),
    ('adam-barnett', 'fridge-jlbwq', 1),
    ('cs131', 'fridge-5ep2w', 4),
    ('fridge-fesjh', 'fridge-eamqk', 3),
    # --- 대파 ---
    ('curry', 'green-zecem', 1),
    ('shabishko', 'onion-leaves-99kwc', 1),
    ('foodanno', 'leek', 1),
    # --- 무 ---
    ('olrnd', 'radish-d1ugk', 1),
    ('dolazy7-gmail-com-3vj05', 'radish', 2),
    ('yeoworm', 'radish-akkqa', 3),
    ('stones-workspace', 'radish-y3ll3', 1),
    ('curry', 'radish-2lmfh', 1),
    # --- 두부 ---
    ('stones-workspace', 'tofu-tdnu5', 1),
    ('graduation-project-3v6ak', 'tofu-nmvji', 1),
    ('chen-xieyu', 'tofu-8punk', 1),
    # --- 우유 ---
    ('mateo-ojeda', 'milk-zum2p', 1),
    ('robocup2022-kogzd', 'milk-i3u51', 2),
    ('bosslnwza', 'milk-fzwnn', 1),
    ('dheebanmilk', 'milk-yxalp', 15),
    ('dolazy7-gmail-com-3vj05', 'milk-acvg1', 2),
    # --- 버섯 ---
    ('test-zq0fq', 'mushroom-3zqg7', 2),
    ('mushroom28', 'mushroom-nksu4', 6),
    ('huurhuntergel', 'mushroom-e8f6k', 3),
    ('yueyue-fob5l', 'mushroom-xtwbh', 1),
    ('mushroom-9xzfs', 'mushroom-jlp5g', 3),
    ('mushroom-r8avj', 'mushroom-syqui', 1),
    ('davifina', 'mushroom-eob88', 2),
    ('genet-xx587', 'mushroom-7whbq', 2),
    # --- 바나나 ---
    ('kipleytaylor', 'banana-rzonz', 1),
    ('jim-pepvn', 'banana-9nmv2', 2),
    ('object-detection-djcen', 'banana-nlcck', 1),
    ('mtfs-workspace-ktra5', 'banana-p8z1q', 5),
    ('train-0beej', 'banana-qlroc', 1),
    # --- 레몬 ---
    ('ai-wfddm', 'lemon-ezutg', 1),
    ('ece4078-mpxui', 'lemon-jlkzg', 2),
    ('mueanfan', 'lemon-orwgb', 2),
    ('riyadh-lakadimu-uhlb5', 'lemon-pknhy', 12),
    ('haniftesting', 'lemon-aer5f', 2),
    ('dcu-oiu8y', 'lemon-gzrja', 2),
    # --- 고추 ---
    ('olrnd', 'green-chili', 1),
    ('abhichal-rajput', 'chili-2jwcr', 1),
    ('gun-yfmsv', 'chili-3apfs', 3),
    ('abhichal-rajput', 'green-chilli', 1),
    # --- 파프리카 (토마토와 구분용) ---
    ('mvi-n1mve', 'bell-pepper-ydvso', 5),
    ('bell-pepper-vpkx5', 'bell-pepper-rld1g', 1),
    ('khuyennguyen', 'bell-pepper-wu9ic', 1),
    ('bellpepper', 'bell-pepper-wxxwg', 4),
    ('thanakrit23', 'bell-pepper-rtzwb', 1),
    # --- 이미 충분한 재료 (상한에 걸리면 자동으로 덜 쓰임) ---
    ('cv-jaka4', 'tomato-detection-ku6fz', 2),
    ('olrnd', 'onion-yhzc7', 1),
    ('fruits-hfscm', 'onion-n278m', 1),
    ('robocup2022-kogzd', 'onion-9renr', 2),
    ('vision-6bf8h', 'onion-yol1h', 1),
    # --- 1·2차에 쓴 데이터 ---
    ('vegetables-19jw7', 'vegetables-2a164', 1),
    ('quiz2pcvk', 'vegetables-8jovr', 1),
    ('robouserflow', 'vegetables-cg8tb', 6),
    ('vegetables-gdugj', 'vegetables-y393s', 1),
    ('vegetables', 'vegetables-el4g6', 1),
    ('09yolosophie', 'vegetables-zvz0j', 3),
    ('dijital-gen', 'vegetables-345nt', 1),
    ('jasur-vkjtq', 'vegetables-tzoje', 1),
    ('leal-jun-fzre6', 'vegetables-9mixn', 1),
    ('jesus-sduc5', 'vegetables-gywcv', 1),
    ('news', 'vegetables-mr0jj', 4),
    ('gian-mwaaw', 'vegetables-fekan', 1),
    ('marvin-sumper', 'vegetables-spshw', 2),
    ('power-k17io', 'vegetables-xrtxa', 1),
    ('myworkspace-tfvkf', 'vegetables-zynnx', 2),
    ('test-lgzia', 'vegetables-7kfdk', 2),
    ('test-on9hk', 'vegetables-kacga', 5),
    ('robosidekick', 'vegetables-2xirf', 3),
    ('shakhob', 'vegetables-e2juh', 1),
    ('monash-university-dkmtq', 'vegetables-guwbg', 3),
    ('workspace-a52as', 'vegetables-zzsys', 1),
    ('2024718', 'vegetables-fkkxs', 6),
    ('raw-food', 'vegetables-mmdn7', 3),
    ('tushar-panwar-xvpdx', 'vegetables-mdufw', 1),
]

# 데이터셋마다 다른 이름(영어 복수형, 스페인어, 터키어, 인도네시아어 등)을 우리 클래스로 자동 연결한다
SYNONYMS = {
    'cabbage': ['cabbage', 'cabbages', 'white_cabbage', 'green_cabbage', 'repollo', 'lahana', 'kubis', 'kol'],
    'green_onion': ['green_onion', 'green_onions', 'spring_onion', 'spring_onions', 'scallion', 'scallions', 'welsh_onion',
                    'daun_bawang', 'leek', 'leeks', 'onion_leaves', 'onion_leaf', 'green_onion_leaves'],
    'onion': ['onion', 'onions', 'red_onion', 'white_onion', 'yellow_onion', 'cebolla', 'sogan', 'bawang_bombay', 'bawang_merah'],
    'potato': ['potato', 'potatoes', 'patata', 'papa', 'patates', 'kentang'],
    'carrot': ['carrot', 'carrots', 'zanahoria', 'havuc', 'wortel'],
    'zucchini': ['zucchini', 'zucchinis', 'courgette', 'calabacin', 'kabak'],
    'cucumber': ['cucumber', 'cucumbers', 'pepino', 'salatalik', 'timun', 'mentimun'],
    'radish': ['daikon', 'white_radish', 'korean_radish'],  # 'radish'(빨간 래디시)는 일부러 뺌
    'spinach': ['spinach', 'espinaca', 'ispanak', 'bayam'],
    'mushroom': ['mushroom', 'mushrooms', 'champignon', 'hongo', 'mantar', 'jamur', 'enoki', 'enoki_mushroom',
                 'shiitake', 'shiitake_mushroom', 'oyster_mushroom', 'button_mushroom', 'king_oyster_mushroom'],
    'tomato': ['tomato', 'tomatoes', 'tomate', 'domates', 'tomat', 'cherry_tomato', 'ripe_tomato', 'red_tomato', 'fresh_tomato'],
    'chili_pepper': ['chili', 'chilli', 'chili_pepper', 'chilli_pepper', 'green_chili', 'green_chilli', 'red_chili',
                     'red_chilli', 'hot_pepper', 'jalapeno', 'cabai', 'cili', 'cabe', 'aji', 'chilies', 'chillies',
                     'green_chilies', 'green_chillies', 'hot_peppers'],
    'garlic': ['garlic', 'ajo', 'sarimsak', 'bawang_putih'],
    'egg': ['egg', 'eggs', 'huevo'],
    'tofu': ['tofu', 'bean_curd', 'tofu_package', 'dubu'],
    'milk': ['milk', 'milk_carton', 'milk_bottle', 'milk_box', 'milk_pack', 'dairy_milk'],
    'apple': ['apple', 'apples', 'manzana', 'elma', 'apel'],
    'banana': ['banana', 'bananas', 'banana_bunch', 'platano', 'muz', 'pisang'],
    'lemon': ['lemon', 'lemons', 'limon', 'lemon_fruit'],
    'bell_pepper': ['bell_pepper', 'bell_peppers', 'capsicum', 'paprika', 'sweet_pepper', 'red_bell_pepper',
                    'green_bell_pepper', 'yellow_bell_pepper', 'orange_bell_pepper'],
    # ---- v4 추가 ----
    'napa_cabbage': ['napa_cabbage', 'chinese_cabbage', 'kimchi_cabbage', 'baechu', 'napa'],
    'pear': ['pear', 'pears', 'asian_pear', 'korean_pear', 'nashi', 'pera', 'armut', 'pir'],
    'mandarin': ['mandarin', 'mandarins', 'mandarin_orange', 'tangerine', 'tangerines', 'clementine', 'clementines',
                 'satsuma', 'mikan', 'jeruk_mandarin'],
    'orange': ['orange', 'oranges', 'naranja', 'portakal', 'jeruk', 'orange_fruit', 'navel_orange'],
    'lime': ['lime', 'limes', 'green_lemon', 'lima', 'jeruk_nipis'],
    'strawberry': ['strawberry', 'strawberries', 'fresa', 'frutilla', 'cilek', 'stroberi'],
    'grape': ['grape', 'grapes', 'green_grape', 'green_grapes', 'red_grape', 'red_grapes', 'grape_bunch', 'uva', 'uvas',
              'uzum', 'anggur'],
    'kiwi': ['kiwi', 'kiwis', 'kiwifruit', 'kiwi_fruit', 'kivi'],
}
# ---- v4 보강: 지난 실행에서 매칭 안 된 이름 중 쓸 수 있는 것 ----
SYNONYMS['cheese'] = ['cheese', 'cheeses', 'cheddar', 'mozzarella', 'sliced_cheese', 'cheese_slice', 'queso', 'peynir', 'keju']
SYNONYMS['sweet_potato'] = ['sweet_potato', 'sweet_potatoes', 'camote', 'batata', 'tatli_patates', 'ubi_jalar', 'goguma']
SYNONYMS['mushroom'] += ['lentinus_edodes',            # 표고
                         'pleurotus_eryngii',          # 새송이
                         'flammulina_velutiper', 'flammulina_velutipes',   # 팽이
                         'hypsizygus_marmoreus',       # 만가닥
                         'pleurotus_ostreatus',        # 느타리
                         'pleurotus_citrinopileatus',  # 노랑느타리
                         'agaricus_bisporus']          # 양송이
# 자동 매칭이 안 된 이름을 직접 연결한다 (None 이면 버림)
EXTRA_MAP = {
    'Fresh_Tomato': 'tomato', 'Fresh_Potato': 'potato', 'Fresh_Carrot': 'carrot',
    'vegetable marrow': 'zucchini', 'chillies': 'chili_pepper',
    'red pepper': 'bell_pepper', 'yellow pepper': 'bell_pepper', 'green pepper': 'bell_pepper',
    'eggplant': 'eggplant', 'Eggplant1': 'eggplant',
    'broccoli': 'broccoli', 'brokoli': 'broccoli',
    'corn': 'corn',
}
# 특정 데이터셋에서만 쓰는 매핑. 'radish'는 빨간 래디시일 수 있어서 무 데이터셋에서만 무로 인정한다
_RADISH = {'radish': 'radish', 'radishes': 'radish', 'daikon': 'radish'}
PER_SOURCE = {
    'radish-d1ugk': _RADISH, 'radish': _RADISH, 'radish-akkqa': _RADISH,
    'radish-y3ll3': _RADISH, 'radish-2lmfh': _RADISH,
}
# 재료별 박스 상한: 이미 이만큼 모인 재료'만' 있는 사진은 건너뛴다 (학습 시간 관리)
MAX_BOXES_PER_CLASS = 12000

def norm(s):
    return re.sub(r'[\s\-]+', '_', str(s).strip().lower())

# classes 는 ② 셀에서 만든 33종 목록을 그대로 쓴다
EN2ID = {c['en']: c['id'] for c in classes}
SYN = {norm(s): en for en, ss in SYNONYMS.items() for s in ss}
SYN.update({norm(k): v for k, v in EXTRA_MAP.items()})

# ---------- 1) 받기 ----------
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ok, failed = [], []
for ws, proj, ver in ROBOFLOW_SOURCES:
    loc = f'/content/datasets/{proj}'
    if glob.glob(f'{loc}/**/data.yaml', recursive=True):
        ok.append(loc); continue                      # 이미 받은 건 건너뜀
    try:
        rf.workspace(ws).project(proj).version(ver).download('yolov8', location=loc)
        ok.append(loc)
    except Exception as e:
        failed.append((proj, str(e)[:120]))

# ---------- 2) 합치기 ----------
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
OUT = Path('/content/merged')
if OUT.exists(): shutil.rmtree(OUT)
for s in ('train', 'val', 'test'):
    (OUT / s / 'images').mkdir(parents=True); (OUT / s / 'labels').mkdir(parents=True)

def to_box(parts):
    """'c x y w h' 는 그대로, 폴리곤 'c x1 y1 x2 y2 ...' 는 감싸는 박스로 바꾼다"""
    v = list(map(float, parts[1:]))
    if len(v) == 4: return v
    if len(v) >= 6 and len(v) % 2 == 0:
        xs, ys = v[0::2], v[1::2]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
    return None

rnd = random.Random(42)
items, seen = [], set()
box_count, unmapped, per_source, dup, capped = Counter(), Counter(), [], 0, 0
for loc in ok:
    y = glob.glob(f'{loc}/**/data.yaml', recursive=True)[0]
    names = yaml.safe_load(open(y))['names']
    names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
    per = {norm(k): v for k, v in PER_SOURCE.get(Path(loc).name, {}).items()}
    def target(n):
        k = norm(n)
        return per[k] if k in per else SYN.get(k)
    cmap = {i: EN2ID[target(n)] for i, n in enumerate(names) if target(n) in EN2ID}
    for i, n in enumerate(names):
        if i not in cmap: unmapped[n] += 0
    used = 0
    for img_dir in Path(loc).rglob('images'):
        lbl_dir = img_dir.parent / 'labels'
        if not lbl_dir.is_dir(): continue
        for img in img_dir.iterdir():
            if img.suffix.lower() not in IMG_EXT: continue
            lines = []
            lbl = lbl_dir / (img.stem + '.txt')
            boxes = []
            if lbl.exists():
                for line in lbl.read_text().split('\n'):
                    p = line.split()
                    if len(p) < 5: continue
                    b = to_box(p)
                    if b is not None: boxes.append((int(float(p[0])), b))
            # 같은 원본 파일명 + 같은 박스 위치 = 포크로 복사된 중복 사진 → 제거
            key = (img.name.split('.rf.')[0].lower(), tuple(sorted(tuple(round(x, 2) for x in b) for _, b in boxes)))
            if key in seen: dup += 1; continue
            mine = [(cmap[c], b) for c, b in boxes if c in cmap]
            for c, b in boxes:
                if c not in cmap and c < len(names): unmapped[names[c]] += 1
            if mine and all(box_count[k] >= MAX_BOXES_PER_CLASS for k, _ in mine):
                capped += 1; continue                       # 이미 충분한 재료만 있는 사진
            for k, b in mine:
                lines.append(f'{k} ' + ' '.join(f'{x:.6f}' for x in b)); box_count[k] += 1
            if not lines and rnd.random() > 0.05: continue  # 우리 재료가 없는 사진은 5%만 남김
            seen.add(key); used += 1
            items.append((img, lines, f'{Path(loc).name}_{len(items)}'))
    per_source.append((Path(loc).name, used, [n for i, n in enumerate(names) if i in cmap]))

rnd.shuffle(items)
n = len(items); n_tr, n_va = int(n * 0.8), int(n * 0.15)
for split, rows in (('train', items[:n_tr]), ('val', items[n_tr:n_tr + n_va]), ('test', items[n_tr + n_va:])):
    for img, lines, stem in rows:
        shutil.copy(img, OUT / split / 'images' / (stem + img.suffix.lower()))
        (OUT / split / 'labels' / (stem + '.txt')).write_text('\n'.join(lines))
yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
                'names': {c['id']: c['en'] for c in classes}}, open(OUT / 'data.yaml', 'w'), sort_keys=False)
DATA_YAML = OUT / 'data.yaml'
shutil.copy('/content/classes.json', OUT / 'classes.json')

# ---------- 3) 결과 ----------
print('\n===== 아래 내용을 복사해서 보내주세요 =====')
print(f'받은 데이터셋 {len(ok)}개, 실패 {len(failed)}개')
for p, e in failed: print('  실패:', p, '-', e)
print(f'사용한 이미지 {n}장 (중복 제거 {dup}장, 상한으로 건너뜀 {capped}장) → train {n_tr} / val {n_va} / test {n - n_tr - n_va}')
print('\n재료별 박스 수:')
for c in classes:
    k = box_count[c['id']]
    note = '  ← AI Hub 합성으로 채움' if c['en'] in ('napa_cabbage',) and k == 0 else '  ← 없음' if k == 0 else '  ← 부족' if k < 150 else ''
    print(f"  {c['id']:>2} {c['ko']}({c['en']}): {k}" + note)
print('\n매칭 안 된 원본 이름(박스 수):')
print('  ' + ', '.join(f'{k}({v})' for k, v in unmapped.most_common(80)) if unmapped else '  없음')

## ④ AI Hub 오려 낸 재료 풀기

In [ ]:
zp = ROOT / 'qc_cutouts.zip'
assert zp.exists(), f'{zp} 가 없다 — 드라이브에 올렸는지 확인'
zipfile.ZipFile(zp).extractall(ROOT)        # → ROOT/cutouts/<클래스>/*.png
print({c.name: len(list(c.glob('*.png'))) for c in CUT.iterdir() if c.is_dir()})

## ⑤ 오려 낸 재료 확인
이상한 사진은 드라이브 `yorijori/qc/cutouts/` 에서 지우면 된다.

In [ ]:
import matplotlib.pyplot as plt
for c in sorted(p for p in CUT.iterdir() if p.is_dir()):
    fs = random.sample(sorted(c.glob('*.png')), min(12, len(list(c.glob('*.png')))))
    fig, ax = plt.subplots(1, len(fs), figsize=(len(fs) * 1.6, 1.9))
    fig.suptitle(c.name)
    for a, f in zip(np.atleast_1d(ax), fs):
        im = cv2.imread(str(f), cv2.IMREAD_UNCHANGED); bg = np.full(im.shape[:2] + (3,), 40, np.uint8)
        al = im[..., 3:] / 255; a.imshow(cv2.cvtColor((im[..., :3] * al + bg * (1 - al)).astype(np.uint8), cv2.COLOR_BGR2RGB)); a.axis('off')
    plt.show()
# 이상한 사진은 드라이브 cutouts 폴더에서 지우면 된다

## ⑥ 배경에 합성 → YOLO 라벨
드라이브 `yorijori/backgrounds/` 에 빈 냉장고·도마 사진을 넣어 두면 배경으로 먼저 쓴다.

In [ ]:
import matplotlib.pyplot as plt
# 속도: 드라이브에서 PNG를 한 장씩 읽고 쓰면 매우 느리다 → 오려 낸 재료를 Colab 디스크로 한 번에 풀고, 합성 결과도 Colab 디스크에 쓴다
LOCAL_CUT = Path('/content/qc_local/cutouts')
if not LOCAL_CUT.exists() or not any(LOCAL_CUT.rglob('*.png')):
    zipfile.ZipFile(ROOT / 'qc_cutouts.zip').extractall('/content/qc_local')
CUT = LOCAL_CUT
CID = {n: i for i, n in enumerate(NAMES)}

# 배경: ① 직접 찍은 냉장고·식탁 사진 폴더(있으면) ② 없으면 무늬·색을 만들어 쓴다
BG_DIR = Path('/content/drive/MyDrive/yorijori/backgrounds')   # 여기에 사진을 넣어 두면 우선 사용
N_SYNTH = 6000            # 만들 합성 사진 수
SYN_OUT = Path('/content/synth')        # Colab 디스크 (빠름). 끝나면 드라이브에 zip으로 백업한다
SZ = 640
random.seed(0); np.random.seed(0)
bgs = sorted(BG_DIR.glob('*.jpg')) + sorted(BG_DIR.glob('*.png')) if BG_DIR.exists() else []
cuts = {c.name: sorted(c.glob('*.png')) for c in CUT.iterdir() if c.is_dir() and c.name in CID}
print('배경 사진', len(bgs), '장 · 재료', {k: len(v) for k, v in cuts.items()})

def make_bg():
    if bgs and random.random() < 0.8:
        im = cv2.imread(str(random.choice(bgs)))
        h, w = im.shape[:2]; s = random.uniform(0.5, 1.0) * min(h, w)
        x, y = random.randint(0, int(w - s)), random.randint(0, int(h - s))
        return cv2.resize(im[y:y + int(s), x:x + int(s)], (SZ, SZ))
    # 냉장고 선반(흰·회색), 도마·식탁(나무색), 장바구니(밝은 단색) 위주의 차분한 색
    kind = random.random()
    if kind < .5:   hsv = (random.randint(0, 179), random.randint(0, 25), random.randint(170, 250))   # 흰·회색
    elif kind < .8: hsv = (random.randint(8, 22), random.randint(60, 150), random.randint(110, 220)) # 나무색
    else:           hsv = (random.randint(0, 179), random.randint(20, 90), random.randint(90, 230))  # 기타 단색
    base = cv2.cvtColor(np.uint8([[hsv]]), cv2.COLOR_HSV2BGR)[0, 0]
    bg = np.full((SZ, SZ, 3), base, np.uint8).astype(np.int16)
    bg += (np.random.randn(SZ, SZ, 1) * random.uniform(3, 14)).astype(np.int16)
    g = np.linspace(-random.randint(0, 50), random.randint(0, 50), SZ)[:, None, None]
    bg = np.clip(bg + (g if random.random() < .5 else g.transpose(1, 0, 2)), 0, 255).astype(np.uint8)
    if random.random() < .4:                                                      # 선반·도마 느낌의 줄무늬
        for y in range(0, SZ, random.randint(60, 200)):
            cv2.line(bg, (0, y), (SZ, y), tuple(int(v) for v in np.clip(base.astype(int) - 40, 0, 255)), random.randint(2, 8))
    return cv2.GaussianBlur(bg, (0, 0), random.uniform(0.5, 2))

def paste(bg, rgba, cx, cy, long_side):
    h, w = rgba.shape[:2]; k = long_side / max(h, w)
    o = cv2.resize(rgba, (max(4, int(w * k)), max(4, int(h * k))), interpolation=cv2.INTER_AREA)
    if random.random() < .5: o = cv2.flip(o, 1)
    ang = random.uniform(-25, 25); M = cv2.getRotationMatrix2D((o.shape[1] / 2, o.shape[0] / 2), ang, 1)
    cos, sin = abs(M[0, 0]), abs(M[0, 1]); nw, nh = int(o.shape[0] * sin + o.shape[1] * cos), int(o.shape[0] * cos + o.shape[1] * sin)
    M[0, 2] += nw / 2 - o.shape[1] / 2; M[1, 2] += nh / 2 - o.shape[0] / 2
    o = cv2.warpAffine(o, M, (nw, nh), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
    rgb = o[..., :3].astype(np.float32) * random.uniform(.7, 1.15) + random.uniform(-15, 15)   # 냉장고 조명 차이
    al = o[..., 3:].astype(np.float32) / 255
    x1, y1 = int(cx - nw / 2), int(cy - nh / 2)
    X1, Y1, X2, Y2 = max(0, x1), max(0, y1), min(SZ, x1 + nw), min(SZ, y1 + nh)
    if X2 - X1 < 8 or Y2 - Y1 < 8: return None
    sx, sy = X1 - x1, Y1 - y1
    a = al[sy:sy + Y2 - Y1, sx:sx + X2 - X1]
    if a.mean() < .15: return None
    # 바닥 그림자 (물건 아래로 살짝 어둡게)
    sh = cv2.GaussianBlur(a[..., 0], (0, 0), 6) * .35
    oy = 6; Ys2 = min(SZ, Y2 + oy); hh = Ys2 - (Y1 + oy)
    if hh > 0: bg[Y1 + oy:Ys2, X1:X2] = (bg[Y1 + oy:Ys2, X1:X2] * (1 - sh[:hh, :, None])).astype(np.uint8)
    reg = bg[Y1:Y2, X1:X2].astype(np.float32)
    bg[Y1:Y2, X1:X2] = np.clip(rgb[sy:sy + Y2 - Y1, sx:sx + X2 - X1] * a + reg * (1 - a), 0, 255).astype(np.uint8)
    ys, xs = np.where(a[..., 0] > .5)
    if len(xs) < 20: return None
    return X1 + xs.min(), Y1 + ys.min(), X1 + xs.max(), Y1 + ys.max()

def iou_cover(b, others):  # b가 앞 물건에 얼마나 가려졌는지(뒤에 붙인 게 위에 온다 → 앞 물건 기준으로 검사)
    for o in others:
        ix = max(0, min(b[2], o[2]) - max(b[0], o[0])); iy = max(0, min(b[3], o[3]) - max(b[1], o[1]))
        if ix * iy > 0.45 * (o[2] - o[0]) * (o[3] - o[1]): return True
    return False

for split in ('train', 'val'):
    (SYN_OUT / 'images' / split).mkdir(parents=True, exist_ok=True); (SYN_OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)
cut_classes = list(cuts)
for n in range(N_SYNTH):
    split = 'val' if n % 10 == 0 else 'train'
    bg = make_bg(); boxes = []
    for _ in range(random.randint(1, 6)):
        c = random.choice(cut_classes); rgba = cv2.imread(str(random.choice(cuts[c])), cv2.IMREAD_UNCHANGED)
        long_side = random.uniform(70, 330)
        tmp = bg.copy()
        b = paste(tmp, rgba, random.uniform(60, SZ - 60), random.uniform(60, SZ - 60), long_side)
        if b is None: continue
        if iou_cover(b, [x[1:] for x in boxes]): continue      # 앞 물건을 절반 가까이 가리면 붙이지 않는다
        bg = tmp
        boxes.append((CID[c], *b))
    if not boxes: continue
    if random.random() < .3: bg = cv2.GaussianBlur(bg, (0, 0), random.uniform(.6, 1.4))       # 흔들림·초점
    if random.random() < .3: bg = cv2.imdecode(cv2.imencode('.jpg', bg, [cv2.IMWRITE_JPEG_QUALITY, random.randint(45, 85)])[1], 1)
    name = f'qc_{n:05d}'
    cv2.imwrite(str(SYN_OUT / 'images' / split / f'{name}.jpg'), bg, [cv2.IMWRITE_JPEG_QUALITY, 92])
    with open(SYN_OUT / 'labels' / split / f'{name}.txt', 'w') as f:
        for c, x1, y1, x2, y2 in boxes:
            f.write(f'{c} {(x1+x2)/2/SZ:.6f} {(y1+y2)/2/SZ:.6f} {(x2-x1)/SZ:.6f} {(y2-y1)/SZ:.6f}\n')
    if n % 500 == 0: print(n, '장')
shutil.make_archive(str(ROOT / 'synth'), 'zip', SYN_OUT)    # 드라이브 백업 → 세션이 끊기면 ⑦에서 자동으로 푼다
print('합성 완료:', len(list((SYN_OUT / 'images' / 'train').glob('*.jpg'))), '/', len(list((SYN_OUT / 'images' / 'val').glob('*.jpg'))))

# 확인용 몇 장
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, f in zip(ax.flat, random.sample(sorted((SYN_OUT / 'images' / 'train').glob('*.jpg')), 8)):
    im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    for line in open(SYN_OUT / 'labels' / 'train' / (f.stem + '.txt')):
        c, x, y, w, h = map(float, line.split()); x1, y1 = int((x - w/2) * SZ), int((y - h/2) * SZ)
        cv2.rectangle(im, (x1, y1), (int((x + w/2) * SZ), int((y + h/2) * SZ)), (0, 255, 0), 2)
        cv2.putText(im, NAMES[int(c)], (x1, max(12, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 255, 0), 1)
    a.imshow(im); a.axis('off')
plt.show()

## ⑦ 학습 설정 파일

In [ ]:
import yaml
SYN_OUT = Path('/content/synth')              # ⑥에서 만든 합성 사진
if not (SYN_OUT / 'images' / 'train').exists() and (ROOT / 'synth.zip').exists():   # 세션이 끊겼으면 드라이브 백업에서 푼다
    zipfile.ZipFile(ROOT / 'synth.zip').extractall(SYN_OUT)
V4_DIR = Path('/content/merged')
assert (V4_DIR / 'data.yaml').exists(), '/content/merged 가 없다 — ③ 셀을 먼저 실행'
assert yaml.safe_load(open(V4_DIR / 'data.yaml'))['names'][32] == 'kiwi', 'merged 가 v3(25종) 기준이다 — ③ 셀을 다시 실행'
data = {
    'path': '/',
    'train': [str(V4_DIR / 'train' / 'images'), str(SYN_OUT / 'images' / 'train')],
    'val':   [str(V4_DIR / 'val' / 'images'), str(SYN_OUT / 'images' / 'val')],
    'names': {i: n for i, n in enumerate(NAMES)},
}
yaml.safe_dump(data, open(ROOT / 'data_v4.yaml', 'w'), allow_unicode=True, sort_keys=False)
json.dump([{'id': i, 'en': en, 'ko': ko} for i, (en, ko) in enumerate(CLASSES_V4)],
          open(ROOT / 'classes_v4.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
n_tr = len(list((V4_DIR / 'train' / 'images').iterdir())) + len(list((SYN_OUT / 'images' / 'train').glob('*.jpg')))
print(open(ROOT / 'data_v4.yaml').read())
print(f'학습 사진 합계 {n_tr}장 (Roboflow + AI Hub 합성)')
DATA_YAML = ROOT / 'data_v4.yaml'
print('DATA_YAML =', DATA_YAML)

## ⑧ 학습

In [ ]:
from ultralytics import YOLO

# v3 가중치에서 이어 학습한다 (처음부터보다 빠르고 기존 25종 실력을 유지한다). 재료가 33종으로 늘어도 YOLO가 마지막 층만 새로 맞춘다
V3_WEIGHTS = f'{WORK}/v3/weights/best.pt'
if not Path(V3_WEIGHTS).exists():                     # 폴더 이름이 다르면 가장 최근 best.pt 를 쓴다
    cands = sorted(glob.glob(f'{WORK}/**/weights/best.pt', recursive=True), key=os.path.getmtime)
    V3_WEIGHTS = cands[-1] if cands else 'yolo26n.pt'
print('시작 가중치:', V3_WEIGHTS)

model = YOLO(V3_WEIGHTS)
model.train(
    data=str(DATA_YAML),
    epochs=25,                      # 사진 5만 장 이상 → T4에서 에폭당 약 10분. v3에서 이어 하므로 25면 충분
    imgsz=640,
    batch=16,
    patience=8,                     # 8 에폭 동안 개선이 없으면 조기 종료
    save_period=1,                  # 매 에폭 드라이브에 저장 → 세션이 끊겨도 이어서 학습
    project=WORK, name='v4', exist_ok=True,
)
BEST = Path(model.trainer.save_dir) / 'weights' / 'best.pt'
print('BEST =', BEST)

In [ ]:
# 세션이 끊겼을 때: 아래 주석을 풀고 실행하면 이어서 학습한다 (먼저 ①②③⑦ 셀을 다시 실행)
# from ultralytics import YOLO
# model = YOLO(f'{WORK}/v4/weights/last.pt'); model.train(resume=True)
# BEST = Path(f'{WORK}/v4/weights/best.pt')

## ⑨ 평가

In [ ]:
from ultralytics import YOLO
m = YOLO(str(BEST))
metrics = m.val(data=str(DATA_YAML), split='val')
print(f"전체 mAP50 = {metrics.box.map50:.3f}")
for i, c in enumerate(classes):
    try:
        print(f"  {c['id']:>2} {c['ko']:<6} mAP50 = {metrics.box.class_result(i)[2]:.3f}")
    except Exception:
        print(f"  {c['id']:>2} {c['ko']:<6} (검증 데이터 없음)")

# 학습에 안 쓴 실제 냉장고 사진으로 눈으로 확인 (드라이브 fridge_yolo/test_photos/ 에 넣어 둔다)
from IPython.display import Image, display
tests = glob.glob(f'{WORK}/test_photos/*.jpg')[:8]
if not tests: print('test_photos 폴더에 사진이 없어 실제 사진 확인은 건너뜀')
for r in (m.predict(tests, conf=0.35, save=True, project=WORK, name='pred_v4', exist_ok=True) if tests else []):
    print(Path(r.path).name, [(classes[int(b.cls)]['ko'], round(float(b.conf), 2)) for b in r.boxes])
for p in sorted(glob.glob(f'{WORK}/pred_v4/*.jpg'))[:8]:
    display(Image(p, width=480))

## ⑩ 웹용 ONNX 내보내기
`nms=False` → 출력 [1, 300, 6] (x1, y1, x2, y2, 점수, 클래스). 웹 데모의 fridge-scan.js 가 이 형식을 쓴다.

In [ ]:
from ultralytics import YOLO
import onnxruntime as ort
onnx_path = YOLO(str(BEST)).export(format='onnx', imgsz=640, nms=False)
s = ort.InferenceSession(onnx_path)
print('입력', s.get_inputs()[0].shape, '출력', s.get_outputs()[0].shape)   # [1,3,640,640] / [1,300,6] 이어야 한다
assert s.get_outputs()[0].shape == [1, 300, 6]

shutil.copy(onnx_path, f'{WORK}/fridge_v4.onnx')
shutil.copy('/content/classes.json', f'{WORK}/classes_v4.json')
print('드라이브 fridge_yolo/ 에 fridge_v4.onnx, classes_v4.json 저장 완료 → 두 파일을 보내 주면 데모에 넣는다')